# Free-text track search with CLAP

Write a prompt, see which tracks of the test set (Traktor rating 4-5 with a colour) come closest.

Kernel: *SongSelector CLAP (venv-clap)* (Select Kernel → Jupyter Kernel…). Registered once with
`.research_cache\venv-clap\Scripts\python -m ipykernel install --user --name songselector-clap --display-name "SongSelector CLAP (venv-clap)"`.

Models: `laion-unfused` (light LAION) and `ms-2023` (Microsoft).
(`laion-music` is left out: in this setup its embeddings collapsed, every text and every track gets nearly the same vector.)

Tip: wording matters. *"This audio is a drum and bass song."* scored clearly higher on DnB tracks than plain *"drum and bass"*.
The embeddings may still be computing: re-run the first cell to load newly finished tracks.

In [ ]:
import os
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / 'pyproject.toml').exists():
    REPO = REPO.parent
os.environ['HF_HOME'] = str(REPO / '.research_cache' / 'hf')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
sys.path.insert(0, str(REPO / 'docs' / 'research' / 'experiments' / 'clap'))

from benchmark_cpu import load_model  # noqa: E402 (needs the sys.path entry above)

CLAP = REPO / '.research_cache' / 'clap'
with sqlite3.connect(CLAP / 'library.sqlite') as connection:
    library = pd.read_sql('SELECT path_key, artist, title, genre, bpm FROM tracks', connection).set_index('path_key')


def unit(matrix):
    return matrix / np.linalg.norm(matrix, axis=-1, keepdims=True)


def track_vectors(model):
    """One vector per track: the average of its window embeddings."""
    with sqlite3.connect(CLAP / f'{model}.sqlite') as connection:
        rows = connection.execute('SELECT path_key, n, dim, vectors FROM windows').fetchall()
    vectors = [unit(np.frombuffer(blob, dtype=np.float32).reshape(n, dim)).mean(0) for _, n, dim, blob in rows]
    return [row[0] for row in rows], unit(np.stack(vectors))


tracks = {model: track_vectors(model) for model in ['laion-unfused', 'ms-2023']}
text_models = {}
{model: len(keys) for model, (keys, _) in tracks.items()}

In [ ]:
def search(prompt, model='laion-unfused', k=10):
    if model not in text_models:
        text_models[model] = load_model(model)
    query = unit(text_models[model].embed_text([prompt])[0])
    keys, vectors = tracks[model]
    scores = vectors @ query
    top = np.argsort(-scores)[:k]
    result = library.loc[[keys[i] for i in top]].reset_index(drop=True)
    result.insert(0, 'score', scores[top].round(3))
    result.index += 1
    return result

In [ ]:
search('dark rolling drum and bass with a heavy reese bassline')

In [ ]:
search('dark rolling drum and bass with a heavy reese bassline', model='ms-2023')